# Meta-Harness++ — Killer Demo

**Goal:** Give MH++ a dataset and an API key. In about 10 minutes it discovers a cheaper, more accurate harness than your hand-written RAG pipeline, and explains *which components* caused the gain.

This notebook walks through the workflow. Each cell is self-contained — run from top to bottom.

## What you'll see

1. Load a 4-class classification benchmark (AG News).
2. Score the seeded baselines (BARE, RAG, CoT-RAG) on the eval set — your floor.
3. Run the MH++ search loop (6 iterations × 8 proposals).
4. Inspect the discovered Pareto frontier, attribution stats, and pair synergy.
5. See which discovered shape strictly Pareto-dominates RAG.

**Cost estimate:** ~$0.05 USD on OpenAI gpt-4.1-nano. Wall time: ~5 minutes.

## 0. Setup

Set your `OPENAI_API_KEY` (or `GEMINI_API_KEY`) in env, then:

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # project root

from meta_harness_plus.tasks import build_agnews_task
from meta_harness_plus.llm.client import HTTPClient
from meta_harness_plus.llm.cache import CachedLLMClient, PromptCache
from meta_harness_plus.llm.predictor import LLMPredictor

task = build_agnews_task()
print(f'Task: {task.name}, {len(task.eval_set)} eval items, {len(task.classes)} classes:')
for c in task.classes: print(f'  - {c}')

## 1. Build the LLM client and a shared cache

The cache makes multi-seed runs fast — every prompt → response pair is SHA-256-keyed and replayed if seen before.

In [ ]:
MODEL = 'gpt-4.1-nano'
raw = HTTPClient(
    api_url='https://api.openai.com/v1/chat/completions',
    api_key=os.environ['OPENAI_API_KEY'],
    model=MODEL,
    timeout_s=300.0,
)
cache = PromptCache(path='runs/demo_notebook/cache.jsonl')
client = CachedLLMClient(raw, cache, model_id=MODEL)
predictor = LLMPredictor(client=client, classes=task.classes, n_samples=1, temperature=0.0, max_tokens=512)
print(f'cache: {len(cache)} pre-loaded keys')

## 2. Score the seeded baselines

These are what a practitioner would write by hand. They become the floor MH++ has to beat.

In [ ]:
from meta_harness_plus.baselines import bare_baseline, rag_baseline, cot_baseline
from meta_harness_plus.scorer import Scorer

scorer = Scorer(task)
for label, h in [('BARE', bare_baseline(task, predictor)),
                  ('RAG',  rag_baseline(task, predictor)),
                  ('CoT-RAG', cot_baseline(task, predictor))]:
    s = scorer.score(h, task.eval_set[:50], n_repeats=2, max_workers=8)
    print(f'{label:8s}  acc={s.accuracy:.3f}  tok={s.tokens:.0f}  lat={s.latency_ms:.0f}ms')

## 3. Run MH++ search

6 iterations × 8 proposals = up to 48 candidate harnesses, with successive halving on a screen subset and full evaluation only for survivors. Drop-one ablation on each Pareto-frontier addition feeds attribution stats back to the LLM proposer.

In [ ]:
from meta_harness_plus.attribution import AttributionTracker
from meta_harness_plus.components import baseline_for
from meta_harness_plus.llm.proposer import LLMProposer
from meta_harness_plus.llm.registry import llm_search_registry
from meta_harness_plus.runner import SearchConfig, SearchRunner

attribution = AttributionTracker(scorer, baseline_for)
registry = llm_search_registry(task, client)
proposer = LLMProposer(client=client, registry=registry, run_dir='runs/demo_notebook/search', temperature=0.5)

runner = SearchRunner(
    task=task, scorer=scorer, proposer=proposer, attribution=attribution,
    config=SearchConfig(
        n_iterations=6, proposals_per_iter=8,
        screen_size=8, full_eval_size=48,
        halving_k0=4, halving_eta=2, halving_final_keep=2,
        eval_repeats=2, attribution_repeats=2, attribution_screen_size=12,
        max_workers=8, screen_seed=0,
        run_dir='runs/demo_notebook/search',
    ),
    seed_harnesses=[bare_baseline(task, predictor), rag_baseline(task, predictor),
                    cot_baseline(task, predictor)],
)
state = runner.run()
print(f'Search complete. {len(state.frontier.entries)} entries on the Pareto frontier.')

## 4. Inspect the discovered frontier

Each row is a harness shape that no other shape strictly dominates.

In [ ]:
for e in sorted(state.frontier.entries, key=lambda x: -x.score.accuracy):
    label = {'cand_0001': '[BARE]', 'cand_0002': '[RAG]', 'cand_0003': '[CoT-RAG]'}.get(e.candidate_id, 'discovered')
    components = ' + '.join(c.get('name', '?') for c in e.meta.get('describe', []))
    print(f'  [{label:>10s}]  acc={e.score.accuracy:.3f}  tok={e.score.tokens:.0f}  lat={e.score.latency_ms:.0f}ms')
    print(f'              {components}')

## 5. Component attribution

Drop-one ablation tells you which kind of component carried its weight.

In [ ]:
for stats in attribution.ranking():
    print(f'  {stats.kind:12s}  mean Δacc = {stats.mean_delta:+.3f}  EWMA Δacc = {stats.ewma_delta:+.3f}  n = {stats.n}')

## 6. Pair synergy (drop-pair attribution)

Drop-pair ablation tells you which component *pairs* only work together — the next level beyond drop-one.

In [ ]:
from meta_harness_plus.synergy import SynergyTracker
synergy = SynergyTracker(scorer, baseline_for)
top = max(state.frontier.entries, key=lambda e: e.score.accuracy)
synergy.analyze(
    candidate_id=top.candidate_id,
    harness=top.meta['harness'],
    examples=task.eval_set[:20],  # smaller subset for the O(K²) drop-pair scan
    full_score=top.score,
)
for stats in synergy.ranking():
    interp = 'super-additive' if stats.mean_synergy > 0.05 else 'redundant' if stats.mean_synergy < -0.05 else 'additive'
    print(f'  {stats.pair[0]} × {stats.pair[1]}: synergy Δacc = {stats.mean_synergy:+.3f} ({interp})')

## 7. Strict Pareto dominance

Did MH++ find a shape that strictly dominates RAG?

In [ ]:
from meta_harness_plus.pareto import dominates
rag_score = next(e for e in state.frontier.entries if e.candidate_id == 'cand_0002').score
discovered = [e for e in state.frontier.entries if e.candidate_id not in ('cand_0001', 'cand_0002', 'cand_0003')]
winners = [e for e in discovered if dominates(e.score, rag_score)]
print(f'Discovered shapes that STRICTLY Pareto-dominate RAG: {len(winners)}')
for w in winners:
    print(f'  acc={w.score.accuracy:.3f}  tok={w.score.tokens:.0f}  lat={w.score.latency_ms:.0f}ms')
    for c in w.meta.get('describe', []):
        print(f'    - {c}')

## What this notebook proved

- ✅ With one API key and ~$0.05 in spend, you can run the full MH++ pipeline.
- ✅ The framework reports the discovered Pareto frontier, per-component attribution, and pair-synergy.
- ✅ Strict Pareto dominance over RAG is checkable, not just a vibe.

For larger benchmarks, multi-seed + bootstrap CIs, ablation studies, and brutal-baselines comparisons, see `examples/` and the `RESULTS_*.md` documents in this repo.